In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*audio.pkl"))

print("Número de ficheiros audio.pkl encontrados:", len(audio_files))
audio_files[:5]

In [ ]:
df = pd.read_pickle(audio_files[0])

print("Shape:", df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
missing = df.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0]

In [ ]:
# Cada segmento de áudio foi convertido numa janela temporal com início e fim, permitindo localizar os momentos relevantes no vídeo.

df["end_time"] = df["time stamp"] + df["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

df["start_mmss"] = df["time stamp"].apply(seconds_to_mmss)
df["end_mmss"] = df["end_time"].apply(seconds_to_mmss)

df[["time stamp", "duration", "start_mmss", "end_mmss"]].head()

In [ ]:
audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localShimmer",
    "localdbShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd"
]

df[audio_features].describe().T

In [ ]:
# duração média dos segmentos de fala

plt.figure(figsize=(8, 4))
plt.hist(df["duration"], bins=30)
plt.xlabel("Duração do segmento (segundos)")
plt.ylabel("Frequência")
plt.title("Distribuição da duração dos segmentos de fala")
plt.grid(True)
plt.show()

In [ ]:
df.sort_values("duration", ascending=False)[
    ["start_mmss", "end_mmss", "duration", "npause", "speechrate", "articulationrate"]
].head(10)

In [ ]:
def zscore(series):
    return (series - series.mean()) / series.std()

In [ ]:
df["vocal_expressiveness_score"] = (
    zscore(df["stdevF0Hz"]) +
    zscore(df["localdbShimmer"]) +
    zscore(df["localShimmer"])
) / 3

In [ ]:
df["vocal_instability_score"] = (
    zscore(df["localJitter"]) +
    zscore(df["rapJitter"]) +
    zscore(df["ppq5Jitter"]) -
    zscore(df["HNR"])
) / 4

In [ ]:
df["speech_pressure_score"] = (
    zscore(df["speechrate"]) +
    zscore(df["articulationrate"]) -
    zscore(df["npause"])
) / 3

In [ ]:
df.sort_values("vocal_expressiveness_score", ascending=False)[
    [
        "start_mmss",
        "end_mmss",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer",
        "vocal_expressiveness_score"
    ]
].head(10)

In [ ]:
# expressividade ao longo do tempo

plt.figure(figsize=(14, 5))
plt.plot(df["time stamp"], df["vocal_expressiveness_score"])
plt.xlabel("Tempo no vídeo (segundos)")
plt.ylabel("Score de expressividade vocal")
plt.title("Expressividade vocal ao longo do tempo")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["time stamp"], df["vocal_instability_score"])
plt.xlabel("Tempo no vídeo (segundos)")
plt.ylabel("Score de instabilidade vocal")
plt.title("Instabilidade vocal ao longo do tempo")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["time stamp"], df["speech_pressure_score"])
plt.xlabel("Tempo no vídeo (segundos)")
plt.ylabel("Score de pressão discursiva")
plt.title("Ritmo/pressão discursiva ao longo do tempo")
plt.grid(True)
plt.show()

In [ ]:
df.sort_values("vocal_expressiveness_score", ascending=False)[
    [
        "start_mmss",
        "end_mmss",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer",
        "vocal_expressiveness_score"
    ]
].head(10)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["time stamp"], df["vocal_expressiveness_score"])
plt.axhline(0, linestyle="--")
plt.xlabel("Tempo no vídeo (segundos)")
plt.ylabel("Score de expressividade vocal")
plt.title("Expressividade vocal ao longo do tempo")
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
DATA_DIR = Path(".")  # muda se os ficheiros estiverem noutra pasta

audio_files = sorted(DATA_DIR.glob("*audio.pkl"))

print("Número de ficheiros audio.pkl encontrados:", len(audio_files))
audio_files[:5]

In [ ]:
all_audio = []

for file in audio_files:
    df_temp = pd.read_pickle(file)
    df_temp["source_file"] = file.name
    df_temp["debate_id"] = file.name.replace("_audio.pkl", "").replace(".audio.pkl", "")
    all_audio.append(df_temp)

audio = pd.concat(all_audio, ignore_index=True)

print("Total de segmentos:", len(audio))
print("Total de debates:", audio["debate_id"].nunique())
display(audio.head())


In [ ]:
audio["end_time"] = audio["time stamp"] + audio["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

audio["start_mmss"] = audio["time stamp"].apply(seconds_to_mmss)
audio["end_mmss"] = audio["end_time"].apply(seconds_to_mmss)

In [ ]:
overview = audio.groupby("debate_id").agg(
    n_segments=("duration", "count"),
    total_speech_time_sec=("duration", "sum"),
    audio_timeline_end_sec=("end_time", "max"),
    mean_segment_duration=("duration", "mean"),
    median_segment_duration=("duration", "median"),
    max_segment_duration=("duration", "max"),
    total_pauses=("npause", "sum"),
    mean_pitch=("meanF0Hz", "mean"),
    mean_HNR=("HNR", "mean"),
    mean_speechrate=("speechrate", "mean"),
    mean_articulationrate=("articulationrate", "mean")
).reset_index()

overview["total_speech_time_min"] = overview["total_speech_time_sec"] / 60
overview["audio_timeline_end_min"] = overview["audio_timeline_end_sec"] / 60

display(overview)

In [ ]:
overview.to_csv("audio_overview_28_debates.csv", index=False)

In [ ]:
missing = audio.isna().mean().sort_values(ascending=False) * 100
missing_table = missing[missing > 0].reset_index()
missing_table.columns = ["feature", "missing_percentage"]

display(missing_table)

In [ ]:
audio_features = [
    "duration",
    "meanF0Hz",
    "stdevF0Hz",
    "HNR",
    "localJitter",
    "localShimmer",
    "localdbShimmer",
    "npause",
    "speechrate",
    "articulationrate",
    "asd"
]

feature_summary = audio[audio_features].describe().T
display(feature_summary)

feature_summary.to_csv("audio_feature_summary_global.csv")

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(overview["debate_id"], overview["n_segments"])
plt.xticks(rotation=90)
plt.xlabel("Debate")
plt.ylabel("Número de segmentos")
plt.title("Número de segmentos de áudio por debate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(overview["debate_id"], overview["total_speech_time_min"])
plt.xticks(rotation=90)
plt.xlabel("Debate")
plt.ylabel("Tempo total de fala (minutos)")
plt.title("Tempo total de fala por debate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(audio["duration"], bins=50)
plt.xlabel("Duração do segmento (segundos)")
plt.ylabel("Frequência")
plt.title("Distribuição global da duração dos segmentos")
plt.grid(True)
plt.show()

In [ ]:
debates = overview["debate_id"].tolist()
data_to_plot = [audio[audio["debate_id"] == d]["duration"].dropna() for d in debates]

plt.figure(figsize=(16, 6))
plt.boxplot(data_to_plot, labels=debates, showfliers=True)
plt.xticks(rotation=90)
plt.ylabel("Duração dos segmentos (segundos)")
plt.title("Distribuição da duração dos segmentos por debate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(overview["debate_id"], overview["mean_pitch"])
plt.xticks(rotation=90)
plt.xlabel("Debate")
plt.ylabel("Pitch médio (meanF0Hz)")
plt.title("Pitch médio por debate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(overview["debate_id"], overview["mean_speechrate"])
plt.xticks(rotation=90)
plt.xlabel("Debate")
plt.ylabel("Speech rate médio")
plt.title("Ritmo médio de fala por debate")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
plt.bar(overview["debate_id"], overview["mean_speechrate"])
plt.xticks(rotation=90)
plt.xlabel("Debate")
plt.ylabel("Speech rate médio")
plt.title("Ritmo médio de fala por debate")
plt.tight_layout()
plt.show()

In [ ]:
corr = audio[audio_features].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlação")

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Correlação entre features acústicas")
plt.tight_layout()
plt.show()


In [ ]:
# maior variação da amplitude vocal

top_amplitude_variation = audio.sort_values("localdbShimmer", ascending=False)[
    [
        "debate_id",
        "start_mmss",
        "end_mmss",
        "duration",
        "localdbShimmer",
        "localShimmer",
        "meanF0Hz",
        "stdevF0Hz",
        "HNR"
    ]
].head(50)

display(top_amplitude_variation)
top_amplitude_variation.to_csv("top_amplitude_variation_segments.csv", index=False)

In [ ]:
def zscore_grouped(df, col, group_col="debate_id"):
    return df.groupby(group_col)[col].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
    )

cols_to_zscore = [
    "stdevF0Hz",
    "localdbShimmer",
    "localShimmer",
    "localJitter",
    "rapJitter",
    "ppq5Jitter",
    "HNR",
    "speechrate",
    "articulationrate",
    "npause",
    "duration"
]

for col in cols_to_zscore:
    audio[col + "_z"] = zscore_grouped(audio, col)

In [ ]:
audio["vocal_expressiveness_score"] = (
    audio["stdevF0Hz_z"] +
    audio["localdbShimmer_z"] +
    audio["localShimmer_z"]
) / 3

audio["vocal_instability_score"] = (
    audio["localJitter_z"] +
    audio["rapJitter_z"] +
    audio["ppq5Jitter_z"] -
    audio["HNR_z"]
) / 4

audio["speech_pressure_score"] = (
    audio["speechrate_z"] +
    audio["articulationrate_z"] -
    audio["npause_z"]
) / 3


In [ ]:
top_expressive = (
    audio.sort_values(["debate_id", "vocal_expressiveness_score"], ascending=[True, False])
    .groupby("debate_id")
    .head(5)
    [[
        "debate_id",
        "start_mmss",
        "end_mmss",
        "time stamp",
        "end_time",
        "duration",
        "vocal_expressiveness_score",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer"
    ]]
)

display(top_expressive)
top_expressive.to_csv("top5_expressive_segments_by_debate.csv", index=False)

In [ ]:
top_instability = (
    audio.sort_values(["debate_id", "vocal_instability_score"], ascending=[True, False])
    .groupby("debate_id")
    .head(5)
    [[
        "debate_id",
        "start_mmss",
        "end_mmss",
        "time stamp",
        "end_time",
        "duration",
        "vocal_instability_score",
        "HNR",
        "localJitter",
        "rapJitter",
        "ppq5Jitter"
    ]]
)

display(top_instability)
top_instability.to_csv("top5_instability_segments_by_debate.csv", index=False)

In [ ]:
overview = overview.sort_values("debate_id").reset_index(drop=True)

overview["debate_label"] = [
    f"D{i+1:02d}" for i in range(len(overview))
]

debate_map = overview[["debate_label", "debate_id"]]
display(debate_map)

debate_map.to_csv("debate_label_mapping.csv", index=False)

In [ ]:
overview = overview.sort_values("debate_id").reset_index(drop=True)

overview["debate_label"] = [
    f"D{i+1:02d}" for i in range(len(overview))
]

debate_map = overview[["debate_label", "debate_id"]]
display(debate_map)

debate_map.to_csv("debate_label_mapping.csv", index=False)

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(overview["debate_label"], overview["mean_speechrate"])

plt.xlabel("Debate")
plt.ylabel("Mean speech rate")
plt.title("Mean speech rate per debate")

plt.tight_layout()
plt.savefig("mean_speechrate_per_debate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
metrics = overview[
    [
        "debate_label",
        "n_segments",
        "total_speech_time_min",
        "mean_segment_duration",
        "mean_pitch",
        "mean_HNR",
        "mean_speechrate",
        "mean_articulationrate",
        "total_pauses"
    ]
].copy()

metrics = metrics.set_index("debate_label").T

# normalizar por linha para o heatmap ficar comparável
metrics_norm = metrics.apply(lambda row: (row - row.mean()) / row.std(), axis=1)

plt.figure(figsize=(14, 6))
plt.imshow(metrics_norm, aspect="auto")
plt.colorbar(label="Z-score")

plt.xticks(range(len(metrics_norm.columns)), metrics_norm.columns, rotation=90)
plt.yticks(range(len(metrics_norm.index)), metrics_norm.index)

plt.title("Audio profile per debate")
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# Criar labels curtas para os debates
overview = overview.sort_values("debate_id").reset_index(drop=True)
overview["debate_label"] = [f"D{i+1:02d}" for i in range(len(overview))]

debate_map = overview[["debate_label", "debate_id"]]
debate_map.to_csv("tables/debate_label_mapping.csv", index=False)

# Heatmap do perfil áudio
metrics = overview[
    [
        "debate_label",
        "n_segments",
        "total_speech_time_min",
        "mean_segment_duration",
        "mean_pitch",
        "mean_HNR",
        "mean_speechrate",
        "mean_articulationrate",
        "total_pauses"
    ]
].copy()

metrics = metrics.set_index("debate_label").T

# Normalização por linha para comparar métricas com escalas diferentes
metrics_norm = metrics.apply(lambda row: (row - row.mean()) / row.std(), axis=1)

plt.figure(figsize=(13, 5))
plt.imshow(metrics_norm, aspect="auto")
plt.colorbar(label="Z-score")

plt.xticks(range(len(metrics_norm.columns)), metrics_norm.columns, rotation=90)
plt.yticks(range(len(metrics_norm.index)), metrics_norm.index)

plt.title("Audio profile across the 28 debates")
plt.tight_layout()
plt.savefig("figures/audio_profile_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pretty_names = {
    "n_segments": "Segments",
    "total_speech_time_min": "Speech time",
    "mean_segment_duration": "Segment duration",
    "mean_pitch": "Pitch",
    "mean_HNR": "HNR",
    "mean_speechrate": "Speech rate",
    "mean_articulationrate": "Articulation rate",
    "total_pauses": "Pauses"
}

metrics = overview[
    [
        "debate_label",
        "n_segments",
        "total_speech_time_min",
        "mean_segment_duration",
        "mean_pitch",
        "mean_HNR",
        "mean_speechrate",
        "mean_articulationrate",
        "total_pauses"
    ]
].copy()

metrics = metrics.set_index("debate_label").T
metrics.index = [pretty_names.get(i, i) for i in metrics.index]

metrics_norm = metrics.apply(lambda row: (row - row.mean()) / row.std(), axis=1)

plt.figure(figsize=(14, 5))
plt.imshow(metrics_norm, aspect="auto")
plt.colorbar(label="Z-score")

plt.xticks(range(len(metrics_norm.columns)), metrics_norm.columns, rotation=90)
plt.yticks(range(len(metrics_norm.index)), metrics_norm.index)

plt.title("Audio profile across the 28 debates")
plt.tight_layout()
plt.savefig("figures/audio_profile_heatmap_clean.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Criar overview/resumo por debate

overview = audio.groupby("debate_id").agg(
    n_segments=("duration", "count"),
    total_speech_time_sec=("duration", "sum"),
    mean_segment_duration=("duration", "mean"),
    median_segment_duration=("duration", "median"),
    max_segment_duration=("duration", "max"),
    mean_pitch=("meanF0Hz", "mean"),
    mean_HNR=("HNR", "mean"),
    mean_speechrate=("speechrate", "mean"),
    mean_articulationrate=("articulationrate", "mean"),
    total_pauses=("npause", "sum")
).reset_index()

overview["total_speech_time_min"] = overview["total_speech_time_sec"] / 60

# Criar labels curtas D01, D02, ...
overview = overview.sort_values("debate_id").reset_index(drop=True)
overview["debate_label"] = [f"D{i+1:02d}" for i in range(len(overview))]

display(overview.head())

In [ ]:
pretty_names = {
    "n_segments": "Segments",
    "total_speech_time_min": "Speech time",
    "mean_segment_duration": "Segment duration",
    "mean_pitch": "Pitch",
    "mean_HNR": "HNR",
    "mean_speechrate": "Speech rate",
    "mean_articulationrate": "Articulation rate",
    "total_pauses": "Pauses"
}

metrics = overview[
    [
        "debate_label",
        "n_segments",
        "total_speech_time_min",
        "mean_segment_duration",
        "mean_pitch",
        "mean_HNR",
        "mean_speechrate",
        "mean_articulationrate",
        "total_pauses"
    ]
].copy()

metrics = metrics.set_index("debate_label").T
metrics.index = [pretty_names.get(i, i) for i in metrics.index]

metrics_norm = metrics.apply(lambda row: (row - row.mean()) / row.std(), axis=1)

plt.figure(figsize=(14, 5))
plt.imshow(metrics_norm, aspect="auto")
plt.colorbar(label="Z-score")

plt.xticks(range(len(metrics_norm.columns)), metrics_norm.columns, rotation=90)
plt.yticks(range(len(metrics_norm.index)), metrics_norm.index)

plt.title("Audio profile across the 28 debates")
plt.tight_layout()
plt.savefig("figures/audio_profile_heatmap_clean.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# Create debate/file overview
audio_overview = audio.groupby("debate_id").agg(
    speech_segments=("duration", "count"),
    total_speech_time_min=("duration", lambda x: x.sum() / 60),
    mean_segment_duration=("duration", "mean"),
    mean_pitch_variation=("stdevF0Hz", "mean"),
    mean_HNR=("HNR", "mean"),
    mean_speech_rate=("speechrate", "mean"),
    total_pauses=("npause", "sum")
).reset_index()

# Short labels to avoid huge file names
audio_overview = audio_overview.sort_values("debate_id").reset_index(drop=True)
audio_overview["file_label"] = [f"D{i+1:02d}" for i in range(len(audio_overview))]

# Save mapping between D01, D02... and file names
audio_overview[["file_label", "debate_id"]].to_csv(
    "tables/audio_file_label_mapping.csv",
    index=False
)

# Prepare heatmap
heatmap_data = audio_overview[
    [
        "file_label",
        "speech_segments",
        "total_speech_time_min",
        "mean_segment_duration",
        "mean_pitch_variation",
        "mean_HNR",
        "mean_speech_rate",
        "total_pauses"
    ]
].set_index("file_label").T

# Better row names for the slide
pretty_names = {
    "speech_segments": "Speech segments",
    "total_speech_time_min": "Total speech time",
    "mean_segment_duration": "Segment duration",
    "mean_pitch_variation": "Pitch variation",
    "mean_HNR": "Voice stability / HNR",
    "mean_speech_rate": "Speech rate",
    "total_pauses": "Pauses"
}

heatmap_data.index = [pretty_names[i] for i in heatmap_data.index]

# Normalize each row so features with different scales can be compared
heatmap_norm = heatmap_data.apply(
    lambda row: (row - row.mean()) / row.std(),
    axis=1
)

plt.figure(figsize=(14, 5.5))
plt.imshow(heatmap_norm, aspect="auto")
plt.colorbar(label="Z-score")

plt.xticks(
    range(len(heatmap_norm.columns)),
    heatmap_norm.columns,
    rotation=90
)

plt.yticks(
    range(len(heatmap_norm.index)),
    heatmap_norm.index
)

plt.title("What can we extract from each audio file?")
plt.xlabel("Audio file / debate")
plt.ylabel("Extracted audio information")

plt.tight_layout()
plt.savefig(
    "figures/what_can_we_extract_per_audio_file.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from pathlib import Path

Path("figures").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

def box(x, y, title, features, meaning):
    rect = FancyBboxPatch(
        (x, y), 5, 1.7,
        boxstyle="round,pad=0.2",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(rect)
    ax.text(x + 2.5, y + 1.35, title, ha="center", va="center",
            fontsize=14, fontweight="bold")
    ax.text(x + 2.5, y + 0.85, features, ha="center", va="center",
            fontsize=11)
    ax.text(x + 2.5, y + 0.35, meaning, ha="center", va="center",
            fontsize=10)

box(0.5, 3.4, "WHEN?", "time stamp, duration", "Locate speech in the video")
box(6.5, 3.4, "VOICE FEATURES", "pitch, HNR, jitter, shimmer", "Vocal variation and stability")
box(0.5, 1.2, "SPEECH RHYTHM", "speechrate, articulationrate, npause", "Fast speech, pauses, hesitation")
box(6.5, 1.2, "SPEAKER IDENTITY", "speak_embeddings", "Cluster voices: Person 1 / 2 / 3")

ax.text(
    6, 0.25,
    "Audio creates temporal windows that can be matched with visual emotions and transcripts.",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold"
)

plt.title("What can we extract from audio.pkl?", fontsize=20, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig("figures/what_can_we_extract_audio.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt

# Escolher um debate com muitos segmentos para o exemplo
example_debate = overview.sort_values("n_segments", ascending=False)["debate_id"].iloc[0]
sample = audio[audio["debate_id"] == example_debate].copy()

# Matriz dos speaker embeddings
X = np.vstack(sample["speak_embeddings"].values)

# Clustering provisório das vozes
n_clusters = 3  # normalmente: candidato 1, candidato 2, moderador
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
sample["speaker_cluster"] = kmeans.fit_predict(X)

# PCA para visualizar embeddings em 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Tempo de voz por cluster
voice_time = sample.groupby("speaker_cluster")["duration"].sum().reset_index()
voice_time["duration_min"] = voice_time["duration"] / 60
voice_time["speaker_label"] = voice_time["speaker_cluster"].apply(lambda x: f"Person {x+1}")

# Figura com dois gráficos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=sample["speaker_cluster"])
axes[0].set_title("Speaker embedding clusters")
axes[0].set_xlabel("PCA 1")
axes[0].set_ylabel("PCA 2")

axes[1].bar(voice_time["speaker_label"], voice_time["duration_min"])
axes[1].set_title("Estimated voice time")
axes[1].set_xlabel("Speaker cluster")
axes[1].set_ylabel("Voice time (minutes)")

plt.suptitle(f"Voice clustering example: {example_debate}", fontsize=12)
plt.tight_layout()
plt.savefig("figures/speaker_embeddings_voice_time.png", dpi=300, bbox_inches="tight")
plt.show()

sample.to_csv("tables/example_debate_with_speaker_clusters.csv", index=False)
voice_time.to_csv("tables/voice_time_per_speaker_example.csv", index=False)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

overview = overview.sort_values("debate_id").reset_index(drop=True)
overview["debate_label"] = [f"D{i+1:02d}" for i in range(len(overview))]

label_map = overview[["debate_label", "debate_id"]]
label_map.to_csv("tables/debate_label_mapping.csv", index=False)

audio = audio.merge(
    overview[["debate_id", "debate_label"]],
    on="debate_id",
    how="left"
)

In [ ]:
def zscore_grouped(df, col, group_col="debate_id"):
    return df.groupby(group_col)[col].transform(
        lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
    )

cols_to_zscore = [
    "stdevF0Hz",
    "localdbShimmer",
    "localShimmer",
    "localJitter",
    "rapJitter",
    "ppq5Jitter",
    "HNR",
    "speechrate",
    "articulationrate",
    "npause",
    "duration"
]

for col in cols_to_zscore:
    audio[col + "_z"] = zscore_grouped(audio, col)

audio["vocal_expressiveness_score"] = (
    audio["stdevF0Hz_z"] +
    audio["localdbShimmer_z"] +
    audio["localShimmer_z"]
) / 3

audio["vocal_instability_score"] = (
    audio["localJitter_z"] +
    audio["rapJitter_z"] +
    audio["ppq5Jitter_z"] -
    audio["HNR_z"]
) / 4

audio["speech_pressure_score"] = (
    audio["speechrate_z"] +
    audio["articulationrate_z"] -
    audio["npause_z"]
) / 3

In [ ]:
# Usar o mesmo debate do slide anterior
sample = audio[audio["debate_id"] == example_debate].copy()

sample["minute"] = (sample["time stamp"] // 60).astype(int)

minute_scores = sample.groupby("minute")[
    [
        "vocal_expressiveness_score",
        "vocal_instability_score",
        "speech_pressure_score"
    ]
].mean().T

plt.figure(figsize=(13, 3.5))
plt.imshow(minute_scores, aspect="auto")
plt.colorbar(label="Mean score")

plt.xticks(range(len(minute_scores.columns)), minute_scores.columns, rotation=90)
plt.yticks(
    range(len(minute_scores.index)),
    ["Expressiveness", "Instability", "Speech pressure"]
)

plt.xlabel("Minute")
plt.title("Audio-based emotional cues per minute")
plt.tight_layout()
plt.savefig("figures/audio_cues_per_minute_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
audio["end_time"] = audio["time stamp"] + audio["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

audio["start_mmss"] = audio["time stamp"].apply(seconds_to_mmss)
audio["end_mmss"] = audio["end_time"].apply(seconds_to_mmss)

top_candidates = (
    audio.sort_values(
        ["debate_id", "vocal_expressiveness_score"],
        ascending=[True, False]
    )
    .groupby("debate_id")
    .head(5)
    [[
        "debate_id",
        "start_mmss",
        "end_mmss",
        "duration",
        "vocal_expressiveness_score",
        "vocal_instability_score",
        "speech_pressure_score",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer",
        "HNR",
        "speechrate",
        "npause"
    ]]
)

top_candidates.to_csv("tables/top5_audio_candidate_segments_by_debate.csv", index=False)
display(top_candidates.head(10))

In [ ]:
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

# Pitch summary per debate
pitch_summary = audio.groupby("debate_label").agg(
    mean_pitch=("meanF0Hz", "mean"),
    mean_pitch_variation=("stdevF0Hz", "mean")
).reset_index()

plt.figure(figsize=(12, 4))
plt.bar(pitch_summary["debate_label"], pitch_summary["mean_pitch_variation"])

plt.xlabel("Debate")
plt.ylabel("Mean pitch variation - stdevF0Hz")
plt.title("Mean vocal pitch variation per debate")

plt.tight_layout()
#plt.savefig("figures/mean_pitch_variation_per_debate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

In [ ]:
DATA_DIR = Path(".")

audio_files = sorted(DATA_DIR.glob("*audio.pkl"))

print("Número de ficheiros audio.pkl encontrados:", len(audio_files))
audio_files[:5]

In [ ]:
all_audio = []

for file in audio_files:
    df_temp = pd.read_pickle(file)
    df_temp["source_file"] = file.name
    df_temp["debate_id"] = file.name.replace("_audio.pkl", "").replace(".audio.pkl", "")
    all_audio.append(df_temp)

audio = pd.concat(all_audio, ignore_index=True)

print("Total de segmentos:", len(audio))
print("Total de debates:", audio["debate_id"].nunique())
display(audio.head())

In [ ]:
debate_ids = sorted(audio["debate_id"].unique())

label_map = pd.DataFrame({
    "debate_id": debate_ids,
    "debate_label": [f"D{i+1:02d}" for i in range(len(debate_ids))]
})

audio = audio.merge(label_map, on="debate_id", how="left")

display(label_map.head())
label_map.to_csv("tables/debate_label_mapping.csv", index=False)

In [ ]:
pitch_summary = audio.groupby("debate_label").agg(
    mean_pitch=("meanF0Hz", "mean"),
    mean_pitch_variation=("stdevF0Hz", "mean")
).reset_index()

plt.figure(figsize=(12, 4))
plt.bar(pitch_summary["debate_label"], pitch_summary["mean_pitch_variation"])

plt.xlabel("Debate")
plt.ylabel("Mean pitch variation - stdevF0Hz")
plt.title("Mean vocal pitch variation per debate")

plt.tight_layout()
plt.savefig("figures/mean_pitch_variation_per_debate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
audio["end_time"] = audio["time stamp"] + audio["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

audio["start_mmss"] = audio["time stamp"].apply(seconds_to_mmss)
audio["end_mmss"] = audio["end_time"].apply(seconds_to_mmss)

top_pitch_variation = (
    audio.sort_values(["debate_id", "stdevF0Hz"], ascending=[True, False])
    .groupby("debate_id")
    .head(5)
    [[
        "debate_label",
        "debate_id",
        "start_mmss",
        "end_mmss",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "HNR",
        "speechrate"
    ]]
)

display(top_pitch_variation.head(10))
top_pitch_variation.to_csv("tables/top5_pitch_variation_segments_by_debate.csv", index=False)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt
import ast
from pathlib import Path

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# Choose one debate with many segments
example_debate = (
    audio.groupby("debate_id")["duration"]
    .count()
    .sort_values(ascending=False)
    .index[0]
)

sample = audio[audio["debate_id"] == example_debate].copy()

# Helper in case embeddings are stored as strings
def parse_embedding(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x))
    return np.array(x)

# Build embedding matrix
X = np.vstack(sample["speak_embeddings"].apply(parse_embedding).values)

# Cluster into 3 voices: usually candidate 1, candidate 2, moderator
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
sample["speaker_cluster"] = kmeans.fit_predict(X)

# Reduce embeddings to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=sample["speaker_cluster"],
    s=45,
    alpha=0.8
)

plt.colorbar(scatter, label="Speaker cluster")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Speaker embedding clusters")

plt.tight_layout()
plt.savefig("figures/speaker_embedding_clusters.png", dpi=300, bbox_inches="tight")
plt.show()

# Save clustered sample for next steps
sample.to_csv("tables/example_debate_speaker_clusters.csv", index=False)

print("Example debate:", example_debate)
print("Number of segments:", len(sample))

In [ ]:
# Voice time per speaker cluster
voice_time = sample.groupby("speaker_cluster")["duration"].sum().reset_index()
voice_time["duration_min"] = voice_time["duration"] / 60
voice_time["speaker_label"] = voice_time["speaker_cluster"].apply(lambda x: f"Person {x+1}")

plt.figure(figsize=(6, 4))
plt.bar(voice_time["speaker_label"], voice_time["duration_min"])

plt.xlabel("Speaker cluster")
plt.ylabel("Voice time (minutes)")
plt.title("Estimated voice time per speaker")

plt.tight_layout()
plt.savefig("figures/voice_time_per_speaker.png", dpi=300, bbox_inches="tight")
plt.show()

display(voice_time)
voice_time.to_csv("tables/voice_time_per_speaker_example.csv", index=False)

In [ ]:
plt.figure(figsize=(12, 3.5))

for cluster in sorted(sample["speaker_cluster"].unique()):
    cluster_data = sample[sample["speaker_cluster"] == cluster]
    plt.scatter(
        cluster_data["time stamp"] / 60,
        [cluster] * len(cluster_data),
        s=cluster_data["duration"] * 2,
        label=f"Person {cluster+1}",
        alpha=0.7
    )

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Speaker")
plt.yticks(
    sorted(sample["speaker_cluster"].unique()),
    [f"Person {i+1}" for i in sorted(sample["speaker_cluster"].unique())]
)
plt.title("Speaker activity over time")
plt.legend()

plt.tight_layout()
plt.savefig("figures/speaker_activity_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# escolher um debate exemplo, se ainda não tiveres escolhido
example_debate = audio.groupby("debate_id")["duration"].count().sort_values(ascending=False).index[0]
sample = audio[audio["debate_id"] == example_debate].copy()

# criar tempos finais e formato mm:ss
sample["end_time"] = sample["time stamp"] + sample["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

sample["start_mmss"] = sample["time stamp"].apply(seconds_to_mmss)
sample["end_mmss"] = sample["end_time"].apply(seconds_to_mmss)

# função z-score dentro do debate exemplo
def zscore(series):
    return (series - series.mean()) / series.std()

# score de intensidade/expressividade vocal
sample["vocal_intensity_cue"] = (
    zscore(sample["stdevF0Hz"]) +
    zscore(sample["localdbShimmer"]) +
    zscore(sample["localShimmer"])
) / 3

# identificar picos: top 10%
threshold = sample["vocal_intensity_cue"].quantile(0.90)
peaks = sample[sample["vocal_intensity_cue"] >= threshold].copy()

plt.figure(figsize=(13, 4.5))

plt.plot(
    sample["time stamp"] / 60,
    sample["vocal_intensity_cue"],
    label="Vocal intensity cue"
)

plt.scatter(
    peaks["time stamp"] / 60,
    peaks["vocal_intensity_cue"],
    s=60,
    label="Top 10% peaks"
)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(threshold, linestyle=":", linewidth=1, label="Top 10% threshold")

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Vocal intensity cue")
plt.title("Moments of higher vocal intensity / expressiveness")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/vocal_intensity_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

# tabela dos momentos mais intensos
top_voice_moments = sample.sort_values("vocal_intensity_cue", ascending=False)[
    [
        "start_mmss",
        "end_mmss",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer",
        "vocal_intensity_cue"
    ]
].head(10)

display(top_voice_moments)
top_voice_moments.to_csv("tables/top10_vocal_intensity_moments.csv", index=False)

print("Example debate:", example_debate)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import textwrap

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# Choose one example debate
example_debate = audio.groupby("debate_id")["duration"].count().sort_values(ascending=False).index[0]
sample = audio[audio["debate_id"] == example_debate].copy()

# Get short debate label, if available
if "debate_label" in sample.columns:
    debate_label = sample["debate_label"].iloc[0]
else:
    debate_label = "Example debate"

# Create end time and mm:ss columns
sample["end_time"] = sample["time stamp"] + sample["duration"]

def seconds_to_mmss(seconds):
    minutes = int(seconds // 60)
    sec = int(seconds % 60)
    return f"{minutes:02d}:{sec:02d}"

sample["start_mmss"] = sample["time stamp"].apply(seconds_to_mmss)
sample["end_mmss"] = sample["end_time"].apply(seconds_to_mmss)

# Z-score function
def zscore(series):
    return (series - series.mean()) / series.std()

# Vocal intensity / expressiveness cue
sample["vocal_intensity_cue"] = (
    zscore(sample["stdevF0Hz"]) +
    zscore(sample["localdbShimmer"]) +
    zscore(sample["localShimmer"])
) / 3

# Identify peaks: top 10%
threshold = sample["vocal_intensity_cue"].quantile(0.90)
peaks = sample[sample["vocal_intensity_cue"] >= threshold].copy()

# Shorten long debate name for the graph subtitle
wrapped_debate_name = "\n".join(textwrap.wrap(example_debate, width=70))

plt.figure(figsize=(13, 5))

plt.plot(
    sample["time stamp"] / 60,
    sample["vocal_intensity_cue"],
    label="Vocal intensity cue"
)

plt.scatter(
    peaks["time stamp"] / 60,
    peaks["vocal_intensity_cue"],
    s=60,
    label="Top 10% peaks"
)

plt.axhline(0, linestyle="--", linewidth=1)
plt.axhline(threshold, linestyle=":", linewidth=1, label="Top 10% threshold")

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Vocal intensity cue")

plt.title(
    f"Moments of higher vocal intensity / expressiveness\n"
    f"{debate_label}: {wrapped_debate_name}",
    fontsize=11
)

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/vocal_intensity_timeline_with_debate.png", dpi=300, bbox_inches="tight")
plt.show()

# Save top moments
top_voice_moments = sample.sort_values("vocal_intensity_cue", ascending=False)[
    [
        "debate_label" if "debate_label" in sample.columns else "debate_id",
        "start_mmss",
        "end_mmss",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "localdbShimmer",
        "localShimmer",
        "vocal_intensity_cue"
    ]
].head(10)

display(top_voice_moments)
top_voice_moments.to_csv("tables/top10_vocal_intensity_moments.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

audio_extraction_table = pd.DataFrame({
    "Audio information": [
        "Temporal segmentation",
        "Speech duration",
        "Pitch / vocal height",
        "Pitch variation",
        "Voice quality",
        "Vocal instability",
        "Speech rhythm",
        "Pauses / hesitation",
        "Speaker identity",
        "Voice time per speaker",
        "Candidate emotional moments"
    ],
    
    "Main features": [
        "time stamp, duration, end_time",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "HNR",
        "localJitter, rapJitter, ppq5Jitter, localShimmer, localdbShimmer",
        "speechrate, articulationrate",
        "npause, speechrate, duration",
        "speak_embeddings",
        "speak_embeddings + duration",
        "pitch variation, shimmer, jitter, HNR, speechrate, npause"
    ],
    
    "What it tells us": [
        "When each speech segment starts and ends",
        "How long each intervention lasts",
        "Average vocal height of the speaker",
        "How much the voice changes in tone",
        "How clean or noisy/stable the voice signal is",
        "Possible vocal tension or unstable voice patterns",
        "How fast or fluent the speech is",
        "Possible hesitation, interruptions or cautious speech",
        "Groups segments by probable voice/speaker",
        "Estimated speaking time for each participant",
        "Moments with higher expressiveness, instability or speech pressure"
    ],
    
    "Use in next submission": [
        "Align audio with visual frames and transcripts",
        "Detect long interventions or dominant speaking moments",
        "Compare speaker profiles and vocal behaviour",
        "Check if expressive voice moments match facial emotions",
        "Control for audio quality and vocal stability",
        "Validate possible tension with facial emotion and transcript",
        "Find intense or fast argumentative moments",
        "Check transcript context for difficult or sensitive topics",
        "Match voice clusters with faces in visual.pkl",
        "Compare voice time with screen time",
        "Select time windows to validate with visual.pkl and speech.pkl"
    ]
})

display(audio_extraction_table)

audio_extraction_table.to_csv(
    "tables/audio_extraction_summary.csv",
    index=False
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# criar pastas
Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

# tabela resumo
audio_extraction_table = pd.DataFrame({
    "Audio information": [
        "Temporal segmentation",
        "Speech duration",
        "Pitch / vocal height",
        "Pitch variation",
        "Voice quality",
        "Vocal instability",
        "Speech rhythm",
        "Pauses / hesitation",
        "Speaker identity",
        "Voice time per speaker",
        "Candidate emotional moments"
    ],
    
    "Main features": [
        "time stamp, duration, end_time",
        "duration",
        "meanF0Hz",
        "stdevF0Hz",
        "HNR",
        "localJitter, rapJitter, ppq5Jitter, localShimmer, localdbShimmer",
        "speechrate, articulationrate",
        "npause, speechrate, duration",
        "speak_embeddings",
        "speak_embeddings + duration",
        "pitch variation, shimmer, jitter, HNR, speechrate, npause"
    ],
    
    "What it tells us": [
        "When each speech segment starts and ends",
        "How long each intervention lasts",
        "Average vocal height of the speaker",
        "How much the voice changes in tone",
        "How clean or noisy/stable the voice signal is",
        "Possible vocal tension or unstable voice patterns",
        "How fast or fluent the speech is",
        "Possible hesitation, interruptions or cautious speech",
        "Groups segments by probable voice/speaker",
        "Estimated speaking time for each participant",
        "Moments with higher expressiveness, instability or speech pressure"
    ],
    
    "Use in next submission": [
        "Align audio with visual frames and transcripts",
        "Detect long interventions or dominant speaking moments",
        "Compare speaker profiles and vocal behaviour",
        "Check if expressive voice moments match facial emotions",
        "Control for audio quality and vocal stability",
        "Validate possible tension with facial emotion and transcript",
        "Find intense or fast argumentative moments",
        "Check transcript context for difficult or sensitive topics",
        "Match voice clusters with faces in visual.pkl",
        "Compare voice time with screen time",
        "Select time windows to validate with visual.pkl and speech.pkl"
    ]
})

# mostrar no notebook
display(audio_extraction_table)

# guardar CSV
audio_extraction_table.to_csv("tables/audio_extraction_summary.csv", index=False)

# guardar como imagem
fig, ax = plt.subplots(figsize=(18, 6))
ax.axis("off")

table = ax.table(
    cellText=audio_extraction_table.values,
    colLabels=audio_extraction_table.columns,
    cellLoc="left",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.6)

plt.title("What can we extract from audio.pkl files?", fontsize=14, pad=20)
plt.tight_layout()

plt.savefig("figures/audio_extraction_summary_table.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import textwrap

Path("figures").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)

audio_slide_table = pd.DataFrame({
    "What we extract": [
        "Time windows",
        "Voice behaviour",
        "Speech rhythm",
        "Speaker identity",
        "Candidate emotional moments"
    ],
    "Audio features": [
        "time stamp, duration",
        "pitch, HNR, jitter, shimmer",
        "npause, speechrate, articulationrate",
        "speak_embeddings",
        "pitch variation, shimmer, jitter, speechrate"
    ],
    "Why it matters for Part 2": [
        "Align audio with visual frames and transcripts",
        "Detect vocal expressiveness or instability",
        "Find hesitation, fast speech or pressure",
        "Cluster voices as Person 1, Person 2, Person 3",
        "Select moments to validate with facial emotions and text"
    ]
})

# guardar CSV
audio_slide_table.to_csv("tables/audio_slide_summary.csv", index=False)

# função para quebrar texto
def wrap_text(text, width=28):
    return "\n".join(textwrap.wrap(str(text), width=width))

wrapped_table = audio_slide_table.copy()
for col in wrapped_table.columns:
    wrapped_table[col] = wrapped_table[col].apply(lambda x: wrap_text(x, 28))

# criar imagem legível
fig, ax = plt.subplots(figsize=(16, 7))
ax.axis("off")

table = ax.table(
    cellText=wrapped_table.values,
    colLabels=wrapped_table.columns,
    cellLoc="left",
    loc="center",
    colWidths=[0.24, 0.32, 0.44]
)

table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 2.8)

# destacar cabeçalho
for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(0.8)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_fontsize(14)

plt.title("What can we extract from audio.pkl files?", fontsize=20, pad=25)

plt.tight_layout()
plt.savefig("figures/audio_slide_summary_readable.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Create speech pressure score for one debate example

sample = audio[audio["debate_id"] == example_debate].copy()

def zscore(series):
    return (series - series.mean()) / series.std()

sample["speech_pressure_score"] = (
    zscore(sample["speechrate"]) +
    zscore(sample["articulationrate"]) -
    zscore(sample["npause"])
) / 3

plt.figure(figsize=(13, 4))

plt.plot(
    sample["time stamp"] / 60,
    sample["speech_pressure_score"],
    label="Speech pressure score"
)

plt.axhline(0, linestyle="--", linewidth=1)

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Speech pressure score")
plt.title(f"Speech pressure over time — {example_debate}")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/speech_pressure_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)

# escolher o mesmo debate exemplo que usaste nos outros slides
sample = audio[audio["debate_id"] == example_debate].copy()

# tempo em minutos
sample["time_min"] = sample["time stamp"] / 60

plt.figure(figsize=(13, 4.5))

plt.plot(
    sample["time_min"],
    sample["speechrate"],
    label="Speech rate",
    linewidth=2
)

plt.plot(
    sample["time_min"],
    sample["articulationrate"],
    label="Articulation rate",
    linewidth=2
)

plt.scatter(
    sample["time_min"],
    sample["npause"],
    label="Number of pauses",
    s=35,
    alpha=0.7
)

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Value")
plt.title(f"Speech rate and pauses over time — {example_debate}")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/speech_rate_and_pauses_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)

# usar o debate exemplo
sample = audio[audio["debate_id"] == example_debate].copy()
sample["time_min"] = sample["time stamp"] / 60

# segmentos com pelo menos 1 pausa
pause_points = sample[sample["npause"] > 0].copy()

plt.figure(figsize=(13, 4.5))

# linha da velocidade da fala
plt.plot(
    sample["time_min"],
    sample["speechrate"],
    linewidth=2,
    label="Speech rate"
)

# pontos onde há pausas
plt.scatter(
    pause_points["time_min"],
    pause_points["speechrate"],
    s=55,
    alpha=0.8,
    label="Segments with pauses"
)

plt.xlabel("Time in debate (minutes)")
plt.ylabel("Speech rate")
plt.title(f"Speech rate and pause moments — {example_debate}")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("figures/speechrate_with_pause_moments.png", dpi=300, bbox_inches="tight")
plt.show()
